In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler,  OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from scipy import stats
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
random_state = 45
target = 'saleprice'

**1---Data Inspection**

In [2]:
df_raw = pd.read_csv('data/train.csv')  #reading the training dataset
df_test = pd.read_csv('data/test.csv')   #reading the test dataset
print('---raw data---')
display(df_raw.head())
print('---raw data shape---')
display(df_raw.shape)
print('---data info---')
display(df_raw.info())
print('---data description---')
display(df_raw.describe().T)

print('---test data---')
display(df_test.head())
print('---test data shape---')
display(df_test.shape)
print('---test data info---')
display(df_test.info())
print('---test data description---')
display(df_test.describe().T)


---raw data---


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


---raw data shape---


(1460, 81)

---data info---
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    1460 non-null   int64  
 18  Ove

...

(1459, 80)

---test data info---
<class 'pandas.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 80 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1459 non-null   int64  
 1   MSSubClass     1459 non-null   int64  
 2   MSZoning       1455 non-null   str    
 3   LotFrontage    1232 non-null   float64
 4   LotArea        1459 non-null   int64  
 5   Street         1459 non-null   str    
 6   Alley          107 non-null    str    
 7   LotShape       1459 non-null   str    
 8   LandContour    1459 non-null   str    
 9   Utilities      1457 non-null   str    
 10  LotConfig      1459 non-null   str    
 11  LandSlope      1459 non-null   str    
 12  Neighborhood   1459 non-null   str    
 13  Condition1     1459 non-null   str    
 14  Condition2     1459 non-null   str    
 15  BldgType       1459 non-null   str    
 16  HouseStyle     1459 non-null   str    
 17  OverallQual    1459 non-null   int64  
 18

None

---test data description---


,count,mean,std,min,25%,50%,75%,max
Id,1459.0,2190.000000,421.321334,1461.0,1825.50,2190.0,2554.50,2919.0
MSSubClass,1459.0,57.378341,42.746880,20.0,20.00,50.0,70.00,190.0
LotFrontage,1232.0,68.580357,22.376841,21.0,58.00,67.0,80.00,200.0
LotArea,1459.0,9819.161069,4955.517327,1470.0,7391.00,9399.0,11517.50,56600.0
OverallQual,1459.0,6.078821,1.436812,1.0,5.00,6.0,7.00,10.0
OverallCond,1459.0,5.553804,1.113740,1.0,5.00,5.0,6.00,9.0
YearBuilt,1459.0,1971.357779,30.390071,1879.0,1953.00,1973.0,2001.00,2010.0
YearRemodAdd,1459.0,1983.662783,21.130467,1950.0,1963.00,1992.0,2004.00,2010.0
MasVnrArea,1444.0,100.709141,177.625900,0.0,0.00,0.0,164.00,1290.0
BsmtFinSF1,1458.0,439.203704,455.268042,0.0,0.00,350.5,753.50,4010.0


**2--Renaming the columns for uniformatiy**

In [3]:
#for training dataset
df_raw.columns = df_raw.columns.str.lower()    #converting the column names into lowercase
df_raw.head()
df = df_raw.copy()
df= df.drop(columns=['id'])
#for test datasets
df_test.columns = df_test.columns.str.lower()  
df_test.head()
df_test= df_test.drop(columns=['id'])
df['saleprice']

0       208500
1       181500
2       223500
3       140000
4       250000
         ...  
1455    175000
1456    210000
1457    266500
1458    142125
1459    147500
Name: saleprice, Length: 1460, dtype: int64

**As our target variable is right_skewed, so applying log1p to reduce the skewness of target variable for better model training**

In [4]:
X_train,y_train = df.drop(columns=['saleprice']),np.log1p(df['saleprice'])
X_test = df_test
print(f'shape of training dataset: {X_train.shape}')
print(f'shape of test dataset: {X_test.shape}')

shape of training dataset: (1460, 79)
shape of test dataset: (1459, 79)


**Changing the type of features based on the info they provide**

In [5]:
print('--mssubclass represents the class of sold property in real-state--\n but here its represented as an integer\n --must be transfered into nominal category, as it represents category of sold property based on style, architecture,..--')
display(df['mssubclass'].unique())
print('--mosold represents the month when the property was sold--\n which is represented as an integer\n --must be transfered into nominal category, as it represents category of month--')
display(df['mosold'].unique())
def change_datatype(df):
    df['mssubclass']=df['mssubclass'].astype(str)  #converting the type of mssubclass into string for nominal category
    df['mosold']=df['mosold'].astype(str)   #same here for month when the house was sold
    return df    

--mssubclass represents the class of sold property in real-state--
 but here its represented as an integer
 --must be transfered into nominal category, as it represents category of sold property based on style, architecture,..--


array([ 60,  20,  70,  50, 190,  45,  90, 120,  30,  85,  80, 160,  75,
       180,  40])

--mosold represents the month when the property was sold--
 which is represented as an integer
 --must be transfered into nominal category, as it represents category of month--


array([ 2,  5,  9, 12, 10,  8, 11,  4,  1,  7,  3,  6])

**1. Classification of datas based on datatypes**

In [6]:
def classify(df):
    target = 'saleprice'
    numerical = [feature for feature in df.select_dtypes(include=np.number).columns if feature!=target]
    categorical = [feature for feature in df.select_dtypes(exclude=np.number).columns if feature!=target]
    return numerical, categorical
   

**2. Checking the null values and duplicate values**

In [7]:
data_report = pd.DataFrame(
    {
        'null_count':df.isna().sum(),  #counting the total number of null values in each feature
        'null_percentage':(df.isna().mean() * 100).round(2)  #and converting them into percentage based on whole training examples
    }
).query('null_count>0').sort_values('null_percentage',ascending=False) #only selecting the datas having null values greater than 0 and sorting them in descending order based on null percentage
print('---The report on null values and null percentage based on every features having null values---')
display(data_report)
print(f'The features having duplicate values: {df.duplicated().sum()}')

---The report on null values and null percentage based on every features having null values---


,null_count,null_percentage
poolqc,1453,99.52
miscfeature,1406,96.30
alley,1369,93.77
fence,1179,80.75
masvnrtype,872,59.73
fireplacequ,690,47.26
lotfrontage,259,17.74
garagetype,81,5.55
garageyrblt,81,5.55
garagefinish,81,5.55


The features having duplicate values: 0


**2.1 Filling null values**

**In this ames house_price prediction datasets, most of the null values in the dataset represents that the house doesnot has this particular feature, so rather than dropping the null values feature or filling them using median or mode, we fill them with None or 0 based on the type of feature, to let the model know that the house doesnot has this particular feature**

**2.2 Every house built has a lotfrontage, so null value of this feature is actually a data_entry mistake and, as the lotfrontage of the houses in the same neighbourhood is almost identical , so null values in this feature can be filled using the median based on the neighbourhood's lotfrontage**

**2.3 Every house built has electricity facility, so null values in them is actually data entry default, so we must fill them based on freqeuntly occuring value**



In [8]:
fill_values = {} 
#in this function, we are passing source so that there won't be dataleakage while filling the null values for test datasets
def fill_na(df,source='train'):
    none_features = ['poolqc', 'miscfeature', 'alley', 'fence', 'masvnrtype',
                     'fireplacequ', 'garagetype', 'garagefinish', 'garagequal',
                     'garagecond', 'bsmtexposure', 'bsmtfintype2', 'bsmtqual', 
                     'bsmtcond', 'bsmtfintype1']
    zero_features = ['masvnrarea', 'garageyrblt']
    for feature in df.columns:
        if feature in none_features:
          df[feature] = df[feature].fillna('None')
        elif feature in zero_features: 
             df[feature] = df[feature].fillna(0)
            
       
    if source=='train':  #only if the passed dataframe is of training dataset, we store the mode of electrical and median of lotfrontage based on the neighborhood
        fill_values['lotfrontage'] = df.groupby('neighborhood')['lotfrontage'].median().to_dict()  #we use the training data's feature median to fill up the null values for lotfrontage
        fill_values['electrical'] = df['electrical'].mode()[0]  #filling the null value of electrical feature with the mode of electrical feature value
    lot_frontage_mean = np.mean(list(fill_values['lotfrontage'].values()))  #calculating the mean of the lotfrontage  
    df['lotfrontage']= df.apply(lambda row: fill_values['lotfrontage'].get(row['neighborhood'],lot_frontage_mean) if pd.isnull(row['lotfrontage']) else row['lotfrontage'],axis = 1) #if the category of the current row isnot found in the dictionary then we fill with the mean of the lotfrontage  
    df['electrical']=df['electrical'].fillna(fill_values['electrical'])
    return df


**3.Initial Feature Selection**

**3.1 Checking the constant features or near constant features** 

In [9]:
def drop_constant_numerical_features(df):
    numerical, categorical = classify(df)
    vt = VarianceThreshold(0.01)
    vt.fit(df[numerical])  #checking the constant features of numerical features based on threshold 0.01
    variance_result = vt.get_support()  #getting the result of checking the passed data with the threshold
    constant_num_features = [col for col,s in zip(df[numerical],variance_result) if not s]  #storing those numerical features whose variance is below 0.01
    df = df.drop(columns = constant_num_features)
    return df
    

def drop_constant_categorical_features(df,y):
    numerical, categorical = classify(df)
    ordinal_categories = {
    'lotshape':      ['IR3', 'IR2', 'IR1', 'Reg'],
    'exterqual':     ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'extercond':     ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'bsmtqual':      ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'bsmtcond':      ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'bsmtexposure':  ['None', 'No', 'Mn', 'Av', 'Gd'],
    'bsmtfintype1':  ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'bsmtfintype2':  ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'heatingqc':     ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'kitchenqual':   ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'functional':    ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'fireplacequ':   ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'garagefinish':  ['None', 'Unf', 'RFn', 'Fin'],
    'garagequal':    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'garagecond':    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'paveddrive':    ['N', 'P', 'Y'],
    'poolqc':        ['None', 'Fa', 'TA', 'Gd', 'Ex'],
    'fence':         ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'electrical':    ['Mix', 'FuseP', 'FuseF', 'FuseA', 'SBrkr']}
    

   
    quasi_constant_cat_features = [feature for feature in categorical if df[feature].value_counts(normalize=True).iloc[0]>0.95]  #if the most frequently occuring category of this current feature is seen in almost 99% of training data, then this feature is considered as constant categorical feature 
    mi_storage = {}
    #instead of directly dropping the near constant categorical features, we must check the statistical relation of these near constant features with the target variable then only we drop those features which has very weak relation with target variable
    for feature in quasi_constant_cat_features:
        if feature in ordinal_categories:
            mapping = {k:i for i,k in enumerate(ordinal_categories[feature])} #creating the dictionary which shows the category as key and its corresponding index as value, here index represents the position of the category in the given order of current nominal categorical feature
            x = df[feature].map(mapping).to_frame(name=feature)   
        else:
            x = df[feature].astype('category').cat.codes.to_frame(name=feature)   #converting the current categorical features value into cat codes, which is done only for nominal categorical features
        mi = mutual_info_regression(x,y.loc[x.index])  #calculating the statistical relationship between categories and the target variable, and here we are using .loc[x.index] for matching the corresponding training data and output variable
        mi_storage[feature] = mi[0]  #storing the feature as key and the mi value as the value in the dict
    features_to_drop = [feature for feature,mi_value in mi_storage.items() if mi_value<0.01]  #we drop those categorical features that have very low statistical relationship with the target variables
    df = df.drop(columns = features_to_drop)   
    return df

    
    
    


**4. Initial Feature Engineering**


**4.1 Creating new features**

In [10]:
def create_new_features(df):
    df['house_age'] = df['yrsold'] - df['yearbuilt']  #creating the age of house
    df['garage_age'] = np.where(df['garageyrblt']!=0,df['yrsold'] - df['garageyrblt'],-1) #creating garage_age, if the garageyrblt is 0, then it means the house didn't had any garage, so we are filling garage age wth -1 in such case
    df['remodeled_age'] = df['yrsold'] - df['yearremodadd'] #calculating the age of house since it was remodeled
    df['has_pool'] = (df['poolarea'] > 0).astype(int) #Has pool?
    df['has_garage'] = (df['garagearea'] > 0).astype(int) #Has garage?
    df['has_fireplace'] = (df['fireplaces'] > 0).astype(int) #Has fireplace?
    df['has_basement'] = (df['totalbsmtsf'] > 0).astype(int)# Has basement?
    df['has_2nd_floor'] = (df['2ndflrsf'] > 0).astype(int) #Has 2nd floor?
    df['has_masonry'] = (df['masvnrarea'] > 0).astype(int) #Has masonry veneer?
    return df



**Dropping the date features, after the creation of related age features, as age fatures are more informative than the data features**

In [11]:
def drop_date_features(df):
    df = df.drop(columns=['yearbuilt','yrsold','garageyrblt','yearremodadd'])
    return df


**5. Feature Selection** 

**a) Numerical Feature Selection**

**5.1 Checking correlation of numerical features with the target variable, and extracting the important features, and dropping the weak numerical features**

In [12]:
def numeric_feature_selection(df,y):
    target = 'saleprice'
    numerical, categorical = classify(df)
    imp_num_features = []
    for feature in numerical:
        corr = df[feature].corr(y.loc[df.index])  #finding the correlation of all the numeric features with the target variable
        if abs(corr)>0.1:
            imp_num_features.append(feature)
    imp_corr_matrix = df[imp_num_features].corr()  #finding the correlation of each important numerical features with eachother
    for i in range(len(imp_corr_matrix.columns)):
     for j in range(i):
        r = imp_corr_matrix.iloc[i,j]  #extracting the correlation value between ith and jth features
        ci = imp_corr_matrix.columns[i]    #extracting the ith feature
        cj = imp_corr_matrix.columns[j]    #extracting the jth feature
        if ci!=cj and abs(r)>0.75 and ci!=target and cj!=target: #if the correlation value between this pair is high, then we remove that feature which has weaker correlation with the target variable
            corr_i_target = df[ci].corr(y.loc[df.index])  #finding the correlation of the ith index feature with the target variable
            corr_j_target = df[cj].corr(y.loc[df.index])  #finding the correlation of the jth index feature with the target variable
            if ci in imp_num_features and  abs(corr_i_target)<abs(corr_j_target):
              imp_num_features.remove(ci)  #removing the weaker feature or the feature which has weaker correlation with the target variable
            elif cj in imp_num_features:
                imp_num_features.remove(cj)
                
    features_to_drop = [feature for feature in numerical if feature not in imp_num_features ]
    df = df.drop(columns = features_to_drop )         
    return df
      
    

**5.2 Checking the multi_collinearity among the important numerical features, to remove the reduntant features**

In [13]:
def drop_multi_collinear(df,y):
    numerical, categorical = classify(df)  #As the multicollinearity is only valid for numerical datas, so we must classify them into numerical and categorical features
    variables = df[numerical].dropna()  #we must drop the null values before checking the variance inflation factor
    vif = pd.DataFrame({
        'features':variables.columns,
        'vif_value':[variance_inflation_factor(variables.values,i) for i in range(variables.shape[1])],
        'corr_with_target':df[numerical].corrwith(y.loc[df.index])  #finding the correlation with the target variable based on their corresponding target 'y'
    }).sort_values('vif_value',ascending=False)
    features_to_drop = vif[(vif['vif_value'] > 10) & (abs(vif['corr_with_target'])<0.1)]['features'].tolist()  #those features which have higher vif value than 10 and very low correlation value of less than 0.1 with the target variable, we drop them 
    df = df.drop(columns = features_to_drop)
    return df

    

**b) Categorical Feature Selection**

**ANOVA TEST for categorical features**

In [14]:
def drop_weak_categorical(df,y):
    numerical,categorical = classify(df)
    anova_report = []
    for feature in categorical:
        groups = [y.loc[group.index].values for _,group in df.groupby(feature)]  #extracting the values of the saleprice based on different categories of current feature, based on the index of the category
        f_stats,p_value = stats.f_oneway(*groups)  #anova test of different saleprice values based on each categories of current feature
        anova_report.append({
            'feature':feature,
            'f_stats':f_stats,
            'p_value':p_value
        })
    total_result_cat = pd.DataFrame(anova_report).sort_values('p_value')
    features_to_drop = total_result_cat[(total_result_cat['p_value']>0.05) | (total_result_cat['p_value'].isna())]['feature'].tolist()
    df = df.drop(columns=features_to_drop)
    return df

    

**6. Feature Engineering pipeline**

In [15]:
stateless_feature_engineering = Pipeline([
    ('change_datatype', FunctionTransformer(change_datatype, validate=False)),  #change the datatypes of feature accordingly
    ('add_features', FunctionTransformer(create_new_features, validate=False)),    #creates the new features denoting the age
    ('drop_dates', FunctionTransformer(drop_date_features, validate=False)),  #dropping the date_based features
])
X_train=stateless_feature_engineering.fit_transform(X_train)  #applying the feature_engineering pipeline to training dataset
X_test = stateless_feature_engineering.fit_transform(X_test)  #also for test dataset





**6.1 Data cleaning process**

In [16]:
X_train = fill_na(X_train,source='train')  #filling the null values of the training datas
X_test = fill_na(X_test,source='test')   #filling  the null values of the test datas
X_train = drop_constant_numerical_features(X_train)  #dropping the constant numerical features of training datas
X_train = drop_constant_categorical_features(X_train,y_train)
X_train = numeric_feature_selection(X_train,y_train)
X_train = drop_multi_collinear(X_train,y_train)  #dropping the features which are higly multi_collinearated
#final training dataset after cleaning the data
X_train = drop_weak_categorical(X_train,y_train) 
X_test = X_test[X_train.columns]  #finally dropping the same fearures based on cleaned training datas

In [17]:
nominal_categories = [
    'mssubclass', 'mszoning', 'alley', 'landcontour', 'lotconfig',
    'neighborhood', 'condition1', 'condition2', 'bldgtype', 'housestyle',
    'roofstyle', 'roofmatl', 'exterior1st', 'exterior2nd', 'masvnrtype',
    'foundation', 'heating', 'centralair', 'garagetype', 'miscfeature',
     'saletype', 'salecondition'
]

In [18]:
ordinal_mapping = {
    'lotshape':      ['IR3', 'IR2', 'IR1', 'Reg'],
    'exterqual':     ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'extercond':     ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'bsmtqual':      ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'bsmtcond':      ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'bsmtexposure':  ['None', 'No', 'Mn', 'Av', 'Gd'],
    'bsmtfintype1':  ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'bsmtfintype2':  ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'heatingqc':     ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'kitchenqual':   ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'functional':    ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'fireplacequ':   ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'garagefinish':  ['None', 'Unf', 'RFn', 'Fin'],
    'garagequal':    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'garagecond':    ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'paveddrive':    ['N', 'P', 'Y'],
    'poolqc':        ['None', 'Fa', 'TA', 'Gd', 'Ex'],
    'fence':         ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'electrical':    ['Mix', 'FuseP', 'FuseF', 'FuseA', 'SBrkr'],
}
ordinal_categories = list(ordinal_mapping.keys())
len(ordinal_categories)

19

**Final classification**

In [19]:
numerical_features,categorical_features = classify(X_train)
nominal_features = [feature for feature in categorical_features if feature in nominal_categories]
ordinal_features =  [feature for feature in categorical_features if feature in ordinal_categories]
len(ordinal_features)

18

**6.2 numerical pipeline**

In [20]:
num_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),  #for filling the null values with the median
    ('scaler',StandardScaler())   
])



**6.3 Ordinal pipeline**

In [21]:
ordinal_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='constant',fill_value='None')),  #for filling the null values with None
    ('encoder', OrdinalEncoder(categories=[ordinal_mapping[col] for col in ordinal_features],handle_unknown='use_encoded_value', unknown_value=-1))
    
])

**6.4 Nominal pipeline**

In [22]:
nominal_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='constant',fill_value='None')),  #for filling the null values with None
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

**6.5 Preprocessor, encoding the training as well as test datasets**


In [23]:
preprocessor = ColumnTransformer([
    ('numerical', num_pipeline,numerical_features),
    ('ordinal', ordinal_pipeline, ordinal_features),
    ('nominal', nominal_pipeline, nominal_features) 
])
X_train_encoded = preprocessor.fit_transform(X_train)  #training the preprocessor using training datas and also encoding the training datas
X_test_encoded = preprocessor.transform(X_test)  #using the trained preprocessor to encode the test datas
X_test_encoded

array([[ 0.43704276,  0.11076257, -0.79515147, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.4816374 ,  0.37584985, -0.07183611, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.16947491,  0.33205282, -0.79515147, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [ 4.00461411,  0.95042275, -0.79515147, ...,  0.        ,
         0.        ,  0.        ],
       [-0.36566079, -0.00759964, -0.79515147, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.16947491, -0.08918038,  0.65147924, ...,  0.        ,
         1.        ,  0.        ]], shape=(1459, 199))

**7. Model Evaluation**

In [31]:
RANDOM_STATE = 42
def evaluate_model(name,model,X,y,cv = 5):
    kf = KFold(n_splits = cv, shuffle=True, random_state = RANDOM_STATE)
    r2 = cross_val_score(model,X,y,cv = kf,scoring='r2')  #calculating the r2 score using the passed model and using kfold splitted datas
    rmse = np.sqrt(-cross_val_score(model,X,y,cv = kf,scoring='neg_mean_squared_error'))
    result = {
        'model':name,
        'r2_score': f'mean: {r2.mean():.4f}, std: {r2.std():.4f}',
        'rmse': f'mean: {rmse.mean():.4f}, std: {rmse.std():.4f}'
    }
    return result
models = [
    ('Ridge Regression', Ridge(alpha=10, random_state=RANDOM_STATE)),
    ('Random Forest',  RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)),
    ('Gradient Boosting',GradientBoostingRegressor(n_estimators=200, random_state=RANDOM_STATE)),
    ('XGBoost', XGBRegressor(n_estimators=200, random_state=RANDOM_STATE, verbosity=0, eval_metric='rmse')),]

results = [evaluate_model(name, model, X_train_encoded, y_train) for name, model in models]
pd.DataFrame(results).set_index('model')    

,r2_score,rmse
model,,
Ridge Regression,"mean: 0.8537, std: 0.0937","mean: 0.1451, std: 0.0365"
Random Forest,"mean: 0.8642, std: 0.0397","mean: 0.1448, std: 0.0170"
Gradient Boosting,"mean: 0.8775, std: 0.0492","mean: 0.1364, std: 0.0216"
XGBoost,"mean: 0.8581, std: 0.0345","mean: 0.1483, std: 0.0130"


**So from the above dataframe, we can observe that Gradient Boosting is the best model based on the overall performance**

**Training the best model**

In [25]:
best_model = GradientBoostingRegressor(n_estimators=200, random_state=RANDOM_STATE)
best_model.fit(X_train_encoded,y_train)  #training the model
y_test_predicted = best_model.predict(X_test_encoded)  #prediction done on test dataset
y_test_predicted

array([11.68397154, 11.94701517, 12.15756595, ..., 11.91646354,
       11.71875128, 12.34166758], shape=(1459,))

**Converting the predicted saleprice into its original range, or reversing the log transformation of saleprice**

In [27]:
reverse_format = np.expm1(y_test_predicted)
reverse_format[:5]

array([118653.54133709, 154354.73230476, 190529.19281725, 187246.31302551,
       196846.26205307])

In [29]:
y_train_predicted = best_model.predict(X_train_encoded)
y_train_predicted[:5]

array([12.23282471, 12.04590312, 12.23765498, 11.96032947, 12.53391458])

In [30]:
y_train[:5]

0    12.247699
1    12.109016
2    12.317171
3    11.849405
4    12.429220
Name: saleprice, dtype: float64